<a href="https://colab.research.google.com/github/Gayathri-rfr/RepoWalker_VectorRAG/blob/RepoWalker_UI/GithubRepoReviewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install langchain-openai langchain-core langchain-community

In [13]:
!git clone https://github.com/tiangolo/typer.git

fatal: destination path 'typer' already exists and is not an empty directory.


In [14]:
!pip install --quiet gradio

In [15]:
!pip install chromadb

#
> Ingesting the repo into chroma




In [16]:
#repo ingestion and retriever
import os
import shutil
from google.colab import userdata
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings


#1.AUthentication and Config

try:
    github_token = userdata.get('GITHUB_TOKEN')
    os.environ["GITHUB_TOKEN"] = github_token
    print("GITHUB_TOKEN loaded successfully.")
except userdata.SecretNotFoundError:
    print("GITHUB_TOKEN not found in Colab Secrets. Defaulting to public access (may hit rate limits).")
    github_token = None

'''

#Clone any repo from github

REPO_OWNER = "tiangolo"
REPO_NAME = "typer"
LOCAL_DIR = f"./{REPO_NAME}"
CHROMA_DIR = "./chroma_db"
for directory in [LOCAL_DIR, CHROMA_DIR]:
    if os.path.exists(directory):
        shutil.rmtree(directory)

print(f"Cloning {REPO_OWNER}/{REPO_NAME}...")
if github_token:
    repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    repo_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

# Clone repo silently
!git clone {repo_url} --quiet
print(" Clone complete.")'''

REPO_NAME = "typer"
LOCAL_DIR = f"./{REPO_NAME}"
loader = DirectoryLoader(
    LOCAL_DIR,
    glob="**/*.py",
    loader_cls=TextLoader,
    show_progress=True
)
documents = loader.load()
print(f"Loaded {len(documents)} source files.")

#language aware chunking  --- needs to be user defined
print("Splitting code tokens systematically...")
splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=1200,
    chunk_overlap=200
)
split_docs = splitter.split_documents(documents)
print(f"Created {len(split_docs)} individual code chunks.")


# vector embedding and chroma storage
CHROMA_DIR = "./chroma_db"
embeddings = OpenAIEmbeddings(model="text-embedding-3-small",
              openai_api_base="https://models.inference.ai.azure.com",
              chunk_size=50) # Added chunk_size to limit batch size for API requests
vector_store = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory=CHROMA_DIR
)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print(f"Success! Vector store initialized and persisted at '{CHROMA_DIR}'.")

GITHUB_TOKEN loaded successfully.


100%|██████████| 634/634 [00:00<00:00, 11257.40it/s]

Loaded 634 source files.
Splitting code tokens systematically...
Created 1390 individual code chunks.


Success! Vector store initialized and persisted at './chroma_db'.


In [17]:
#ytesting the embeddings:
test_text = "def hello_world(): print('Hello from GitHub Models!')"
vector = embeddings.embed_query(test_text)

print(f" Success! Generated embedding vector.")
print(f"Vector Dimensions: {len(vector)}")
print(f"Sample values: {vector[:5]}")

 Success! Generated embedding vector.
Vector Dimensions: 1536
Sample values: [-0.015167236328125, -0.01873779296875, 0.046356201171875, 0.0031719207763671875, 0.0030422210693359375]




> app.py



In [18]:
#app.py
from langchain_openai import ChatOpenAI,OpenAI
from google.colab import userdata
import os
from openai import OpenAI
from typing_extensions import TypedDict,List
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import add_messages, START, StateGraph, END # Re-import add_messages
from typing import Annotated # Import Annotated

#openAI token
token = userdata.get('GITHUB_TOKEN')
os.environ["OPENAI_API_KEY"] = token

# Initialize the language model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0,api_key=token,
   base_url="https://models.inference.ai.azure.com")

#Agent structure
class AgentState(TypedDict):
    query: str
    context: List[str]
    messages: Annotated[List[BaseMessage], add_messages] # Reverted to use Annotated with add_messages
    iterations: int

def retrieve_code(state: AgentState):
    """Fetches code from Chroma."""
    query = state["query"]
    # If the model updated the message history with a refined search phrase, use it
    if state["messages"] and isinstance(state["messages"][-1], AIMessage):
        query = state["messages"][-1].content

    docs = retriever.invoke(query)
    formatted_context = [f"File: {d.metadata.get('source')}\n\n{d.page_content}" for d in docs]

    return {
        "context": list(set(state["context"] + formatted_context)),
        "iterations": state["iterations"] + 1
    }

def evaluate_and_answer(state: AgentState):
    """LLM looks at the context and decides to answer or ask for more details."""
    system_prompt = (
        "You are an expert code assistant. Look at the retrieved context from the repository.\n"
        "If you have enough information to confidently answer the user's question, provide the answer.\n"
        "If you need to see another file or require more context, write a specific search query 'NEED_MORE_CONTEXT:' targeting that missing code "
    )

    context_str = "\n\n---\n\n".join(state["context"])

    # Construct the current turn's HumanMessage
    current_human_message_content = f"User Question: {state['query']}\n\nRetrieved Code Context:\n{context_str}"
    current_human_message = HumanMessage(content=current_human_message_content)

    # All messages for the LLM call: System, accumulated history, and current Human message
    messages_for_llm = [SystemMessage(content=system_prompt)] + state["messages"] + [current_human_message]

    # Call the OpenAI-compatible model with error handling
    try:
        response_ai_message = llm.invoke(messages_for_llm)
    except Exception as e:
        print(f"Error during LLM invoke: {e}")
        # Create a fallback AI message if the LLM call fails
        response_ai_message = AIMessage(content=f"LLM call failed: {e}. Cannot generate response.")

    # Ensure current_human_message is always a BaseMessage (should be by construction)
    if not isinstance(current_human_message, BaseMessage):
        current_human_message = HumanMessage(content="[Fallback] User query message could not be properly constructed.")

    # Return *only* the new messages generated in this step. add_messages will merge them.
    return {"messages": [current_human_message, response_ai_message]}

def should_continue(state: AgentState):
    """Conditional edge determining whether to loop or stop."""
    # Safeguard against infinite loops
    if state["iterations"] >= 5: # Increased from 3 to 5 for more turns
        return "end"

    # Check if messages list is not empty before accessing the last element
    if not state["messages"]:
        # This should ideally not happen if add_messages works correctly,
        # but provides an extra layer of safety.
        return "end"

    last_message = state["messages"][-1].content
    if "NEED_MORE_CONTEXT:" in last_message:
        return "continue"
    return "end"


In [19]:
# Initialize Workflow
workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("retrieve", retrieve_code)
workflow.add_node("analyze_and_respond", evaluate_and_answer)

# Set Entry Point
workflow.set_entry_point("retrieve")

# Add Edges
workflow.add_edge("retrieve", "analyze_and_respond")

# Add Conditional Edges
workflow.add_conditional_edges(
    "analyze_and_respond",
    should_continue,
    {
        "continue": "retrieve",
        "end": END
    }
)

# Compile Graph
app = workflow.compile()


In [20]:
# Initialize execution state
initial_input = {
 #   "query": "How does Typer handle custom commands or application callbacks internally?",
    "query": "How many types of implementaton is possible?",
    "context": [],
    "messages": [],
    "iterations": 0
}

# Run the graph
output = app.invoke(initial_input)

print("\n================== FINAL RESPONSE ==================\n")
print(output["messages"][-1].content)



================== FINAL RESPONSE ==================

Based on the retrieved code context, it appears that the implementation types being handled are related to type annotations in Python, specifically within the Typer library. The code mentions handling for:

1. **Union types**: The code checks for `Union` types and asserts that only one type is supported, indicating that Typer currently does not support multiple types in a union.
2. **List types**: The code handles lists, asserting that complex sub-types are not supported.
3. **Tuple types**: The code also handles tuples, again asserting that complex sub-types are not supported.

From this, we can infer that the types of implementations possible in this context are:

- Single types (e.g., `int`, `str`, etc.)
- Lists of a single type (e.g., `List[int]`)
- Tuples of multiple single types (e.g., `Tuple[int, str]`)

However, it explicitly states that complex sub-types for lists and tuples are not supported, and unions are limited to a s

In [25]:
import gradio as gr

# Define a function to interact with the LangChain graph
def chat_with_graph(message, history):
    # Initialize execution state for each new chat session
    # Note: For persistent chat history, you might need to manage `messages` more robustly
    # or pass `history` to the graph's initial_input after converting it to BaseMessage format.

    # Convert Gradio history to LangChain messages format
    lc_messages = []
    for human, ai in history:
        lc_messages.append(HumanMessage(content=human))
        lc_messages.append(AIMessage(content=ai))

    # Add the current human message
    lc_messages.append(HumanMessage(content=message))

    initial_input = {
        "query": message,
        "context": [],
        "messages": lc_messages, # Pass the accumulated messages
        "iterations": 0
    }

    # Run the graph with the current input
    output = app.invoke(initial_input)

    # The last message in the output is the AI's response to the current query
    response_message = output["messages"][-1].content
    return response_message

# Create and launch the Gradio ChatInterface
chat_interface = gr.ChatInterface(
    fn=chat_with_graph,
    chatbot=gr.Chatbot(height=500),
    textbox=gr.Textbox(placeholder="Ask me about the Typer repository...", container=False, scale=7),
    title="Typer Repository Code Assistant",
    description="Ask questions about the Typer codebase, and I'll use LangGraph to find answers.",
    theme="soft",
    examples=["How does Typer handle custom commands?", "What is `typer.run` used for?", "Show me an example of a simple Typer app."],
    cache_examples=False
)

chat_interface.launch(debug=True)

/tmp/ipykernel_10849/4101473686.py:35: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot=gr.Chatbot(height=500),
/tmp/ipykernel_10849/4101473686.py:35: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot=gr.Chatbot(height=500),
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:330: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'tuples', will be used.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://96a573de2e3033ee7c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://96a573de2e3033ee7c.gradio.live


### 1. Consolidate Application Code into `app.py`

For deployment, it's best practice to consolidate your application's Python logic into a single script, usually `app.py`. This file will include the ChromaDB loading, LLM and embeddings initialization, LangGraph agent definition, and the Gradio interface setup.

In [26]:
%%writefile app.py

import os
import gradio as gr
from typing import Annotated, List, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langgraph.graph import add_messages, START, StateGraph, END

# --- Environment Variable Setup ---
# For Cloud Run, secrets like OPENAI_API_KEY should be passed as environment variables.
# For local testing within Colab, we'll try to fetch it from Colab Secrets if not set as an environment variable.
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    try:
        from google.colab import userdata
        openai_api_key = userdata.get('OPENAI_API_KEY')
        if not openai_api_key:
            raise ValueError("OPENAI_API_KEY not found in Colab Secrets or environment variables.")
    except ImportError:
        raise ValueError("OPENAI_API_KEY not found in environment variables. Set it for Cloud Run deployment.")

# --- LLM and Embeddings Initialization ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=openai_api_key,
                 base_url="https://models.inference.ai.azure.com")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small",
                              openai_api_base="https://models.inference.ai.azure.com",
                              chunk_size=50)

# --- ChromaDB Loading ---
CHROMA_DIR = "./chroma_db"
# Ensure the chroma_db directory exists and is populated
if not os.path.exists(CHROMA_DIR):
    raise FileNotFoundError(f"ChromaDB directory not found at {CHROMA_DIR}. Please run the ingestion notebook first.")

vector_store = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings # Must pass the same embedding function used to create it
)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# --- Agent State and Functions ---
class AgentState(TypedDict):
    query: str
    context: List[str]
    messages: Annotated[List[BaseMessage], add_messages]
    iterations: int

def retrieve_code(state: AgentState):
    query = state["query"]
    if state["messages"] and isinstance(state["messages"][-1], AIMessage):
        query = state["messages"][-1].content
    docs = retriever.invoke(query)
    formatted_context = [f"File: {d.metadata.get('source')}\n\n{d.page_content}" for d in docs]
    return {
        "context": list(set(state["context"] + formatted_context)),
        "iterations": state["iterations"] + 1
    }

def evaluate_and_answer(state: AgentState):
    system_prompt = (
        "You are an expert code assistant. Look at the retrieved context from the repository.\n"
        "If you have enough information to confidently answer the user's question, provide the answer.\n"
        "If you need to see another file or require more context, write a specific search query 'NEED_MORE_CONTEXT:' targeting that missing code "
    )
    context_str = "\n\n---\n\n".join(state["context"])
    current_human_message_content = f"User Question: {state['query']}\n\nRetrieved Code Context:\n{context_str}"
    current_human_message = HumanMessage(content=current_human_message_content)
    messages_for_llm = [SystemMessage(content=system_prompt)] + state["messages"] + [current_human_message]
    try:
        response_ai_message = llm.invoke(messages_for_llm)
    except Exception as e:
        response_ai_message = AIMessage(content=f"LLM call failed: {e}. Cannot generate response.")
    if not isinstance(current_human_message, BaseMessage):
        current_human_message = HumanMessage(content="[Fallback] User query message could not be properly constructed.")
    return {"messages": [current_human_message, response_ai_message]}

def should_continue(state: AgentState):
    if state["iterations"] >= 5:
        return "end"
    if not state["messages"]:
        return "end"
    last_message = state["messages"][-1].content
    if "NEED_MORE_CONTEXT:" in last_message:
        return "continue"
    return "end"

# --- Workflow Definition ---
workflow = StateGraph(AgentState)
workflow.add_node("retrieve", retrieve_code)
workflow.add_node("analyze_and_respond", evaluate_and_answer)
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "analyze_and_respond")
workflow.add_conditional_edges(
    "analyze_and_respond",
    should_continue,
    {"continue": "retrieve", "end": END}
)
app = workflow.compile()

# --- Gradio Interface ---
def chat_with_graph(message, history):
    lc_messages = []
    for human, ai in history:
        lc_messages.append(HumanMessage(content=human))
        lc_messages.append(AIMessage(content=ai))
    lc_messages.append(HumanMessage(content=message))

    initial_input = {
        "query": message,
        "context": [],
        "messages": lc_messages,
        "iterations": 0
    }
    output = app.invoke(initial_input)
    response_message = output["messages"][-1].content
    return response_message

chat_interface = gr.ChatInterface(
    fn=chat_with_graph,
    chatbot=gr.Chatbot(height=500, type='messages'), # Using type='messages' as recommended by Gradio
    textbox=gr.Textbox(placeholder="Ask me about the Typer repository...", container=False, scale=7),
    title="Typer Repository Code Assistant",
    description="Ask questions about the Typer codebase, and I'll use LangGraph to find answers.",
    theme="soft",
    examples=["How does Typer handle custom commands?", "What is `typer.run` used for?", "Show me an example of a simple Typer app."],
    cache_examples=False
)

# Launch Gradio app on 0.0.0.0 and dynamically assigned port for Cloud Run
if __name__ == "__main__":
    chat_interface.launch(server_name="0.0.0.0", server_port=int(os.environ.get("PORT", 7860)))

Writing app.py


### 2. Create `requirements.txt`

This file lists all the Python dependencies your application needs. Cloud Run will use this to install the necessary libraries.

In [27]:
%%writefile requirements.txt
langchain-openai
langchain-core
langchain-community
gradio
chromadb
langgraph
openai
tiktoken
pydantic>=2.7.4
typing-extensions


Writing requirements.txt


### 3. Create `Dockerfile`

The `Dockerfile` defines how your application's environment will be built. It specifies the base image, copies your code and dependencies, and sets the command to run your application.

In [28]:
%%writefile Dockerfile

# Use an official Python runtime as a parent image
FROM python:3.10-slim-buster

# Set the working directory in the container
WORKDIR /app

# Copy the requirements file into the container at /app
COPY requirements.txt .

# Install any needed packages specified in requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

# Copy the local chroma_db directory into the container
# This assumes the chroma_db directory is created by the ingestion steps
# and is available in the context when building the Docker image.
# Ensure your 'chroma_db' directory is in the same folder as your Dockerfile and app.py
COPY chroma_db ./chroma_db

# Copy the app.py file into the container at /app
COPY app.py .

# Expose the port that Gradio will run on
# Cloud Run typically expects services to listen on the port specified by the PORT environment variable
ENV PORT 8080
EXPOSE 8080

# Run app.py when the container launches
CMD ["python", "app.py"]

Writing Dockerfile


### 4. Deploy to Google Cloud Run

Once you have created `app.py`, `requirements.txt`, and `Dockerfile` in the same directory (along with your `chroma_db` folder), you can deploy your application to Google Cloud Run. You will need to have the Google Cloud SDK installed and authenticated.

First, make sure your `chroma_db` directory is populated from the previous ingestion steps.

Then, open a terminal in the directory containing these files and run the following commands:

1.  **Build and Deploy:**

    Replace `YOUR_SERVICE_NAME` with a unique name for your service and `YOUR_REGION` with an appropriate Google Cloud region (e.g., `us-central1`). You must also set the `OPENAI_API_KEY` as an environment variable for your Cloud Run service. You can use the `--set-env-vars` flag.


In [29]:
#!gcloud run deploy YOUR_SERVICE_NAME \
#    --source . \
#    --region YOUR_REGION \
#    --allow-unauthenticated \
#    --set-env-vars OPENAI_API_KEY=$OPENAI_API_KEY \
#    --min-instances 0 \
#    --max-instances 1 \
#    --cpu 1 \
#    --memory 2Gi \
#    --timeout 300

print("Replace 'YOUR_SERVICE_NAME' and 'YOUR_REGION' with your actual values.")
print("Make sure the OPENAI_API_KEY environment variable is set in your terminal session before running this command, or specify the key directly.")
print("This command assumes you are running it from a local terminal where you have the Google Cloud SDK installed and authenticated.")


Replace 'YOUR_SERVICE_NAME' and 'YOUR_REGION' with your actual values.
Make sure the OPENAI_API_KEY environment variable is set in your terminal session before running this command, or specify the key directly.
This command assumes you are running it from a local terminal where you have the Google Cloud SDK installed and authenticated.
